# EV Policy Assistant

## Stage 1: environment checks

Stage 1 setup checks. A separate Stage 2 worked example follows below. Adapted from the course repository at commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`: Lab 4 cells 6–8, 38, 65, 77, 106 and 150; Exercise 2 cell 4. AI assistance adapted these setup checks; no policy-answering pipeline is implemented here.

Run from the project folder with the project’s Python 3.12 environment. Put your Groq key in the local `.env` file. Notebook outputs should be cleared before committing.

In [ ]:
import os
import sys
import math
from importlib.metadata import version
from dotenv import load_dotenv
import gradio as gr
from langchain.chat_models import init_chat_model
from langchain.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)

assert sys.version_info[:2] == (3, 12), "Select the project Python 3.12 kernel."
print("Python:", sys.version.split()[0])
for package in ["langchain", "langchain-chroma", "langchain-ollama", "langchain-groq", "gradio", "pypdf", "unstructured"]:
    print(package, version(package))

### Local embeddings

Ollama must be running with `nomic-embed-text` available. This checks one query; it does not build an index.

In [ ]:
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
query_embedding = embeddings_model.embed_query("EV policy setup check")
assert len(query_embedding) > 0
assert all(math.isfinite(value) for value in query_embedding)
print("Embedding dimensions:", len(query_embedding))

### Groq connection

This sends a small test prompt to Groq and uses the account’s API allowance. A missing key stops the check; it is not a successful connection test.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    raise ValueError("Add GROQ_API_KEY to the local .env file and rerun the setup cells.")

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0, max_tokens=256, timeout=30, max_retries=0)
response = llm.invoke("Reply with only OK.")
assert response.content, "Groq returned an empty response."
print(response.content)

## Stage 2: page-loading example

AI-assisted worked example for one page, within the agreed helper-only scope. It starts Stage 2; it is not the completed ingestion pipeline.

Reuses Exercise 2 cell 9's `Document(page_content=..., metadata=...)` pattern and Lab 4 cells 65–66's load-then-inspect sequence at course commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`. `pypdf.PdfReader` is an added page-preserving alternative, not the loader used in the course. Physical page 18 is Python page index 17. `conditions_pdf_page` links to page 19 of the same source; page 18 remains the citation for this text.

These cells run independently of the model setup above: no API key, Groq call or Ollama request is needed. Original PDFs remain the citation targets. Manual review is deferred; the example retains the manifest's unverified status.


In [ ]:
import json
import hashlib
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

source_manifest = json.loads(Path('data/source_manifest.json').read_text())
policy_source = next(source for source in source_manifest['sources']
                     if source['source_id'] == 'maharashtra_policy_2025-05-23')
policy_path = Path(policy_source['filename'])
assert hashlib.sha256(policy_path.read_bytes()).hexdigest() == policy_source['sha256']

pdf_page = 18
policy_text = PdfReader(policy_path).pages[pdf_page - 1].extract_text()
if not policy_text or not policy_text.strip():
    raise ValueError(f'No readable text on PDF page {pdf_page}.')

policy_document = Document(page_content=policy_text, metadata={
    'state': policy_source['state'],
    'policy_year': policy_source['policy_year'],
    'source_id': policy_source['source_id'],
    'source': policy_source['filename'],
    'document_title': policy_source['document_title'],
    'official_url': policy_source['official_url'],
    'document_date': policy_source['document_date'],
    'pdf_page': pdf_page,
    'conditions_pdf_page': 19,
    'team_verified': policy_source['team_verified'],
    'accepted_for_ingestion': policy_source['accepted_for_ingestion'],
    'current_benefit_availability': policy_source['current_benefit_availability'],
    'current_entitlement_answers_allowed': policy_source['current_entitlement_answers_allowed'],
    'verification_cutoff': source_manifest['verification_cutoff'],
})

print(policy_document.metadata)
print(policy_document.page_content)


In [ ]:
assert policy_document.metadata['pdf_page'] == 18
assert policy_document.metadata['conditions_pdf_page'] == 19
assert policy_document.metadata['team_verified'] is False
assert policy_document.metadata['accepted_for_ingestion'] is False
assert policy_document.metadata['current_entitlement_answers_allowed'] is False
assert 'Table 2: Demand Incentives for EVs' in policy_document.page_content
assert 'Maximum' in policy_document.page_content and 'Rupees' in policy_document.page_content
print('One candidate page loaded with provenance. Stage 2 is still in progress.')


### Load the English policy pages

This bounded extension of the worked example creates ten page records. It uses Exercise 2 cell 9's list/`Document` pattern with the page-preserving reader above. Run the Stage 2 example first; the Stage 1 model cells are still unnecessary.

Copy the shared metadata, then set the physical page for each record. The conditions-page link belongs only to Table 2 on page 18; copying it onto every page would create a wrong relationship. OCR loading, other cross-page relationships and splitting remain the team's next implementation work.


In [ ]:
policy_reader = PdfReader(policy_path)
assert len(policy_reader.pages) == policy_source['pdf_page_count']

policy_metadata = policy_document.metadata.copy()
policy_metadata.pop('pdf_page')
policy_metadata.pop('conditions_pdf_page')

policy_documents = []
for pdf_page in policy_source['candidate_pdf_pages']:
    page_text = policy_reader.pages[pdf_page - 1].extract_text()
    if not page_text or not page_text.strip():
        raise ValueError(f'No readable text on PDF page {pdf_page}.')

    page_metadata = policy_metadata.copy()
    page_metadata['pdf_page'] = pdf_page
    if pdf_page == 18:
        page_metadata['conditions_pdf_page'] = 19

    policy_documents.append(Document(page_content=page_text, metadata=page_metadata))

print('English policy pages loaded:', len(policy_documents))
print('Physical pages:', [doc.metadata['pdf_page'] for doc in policy_documents])


In [ ]:
assert len(policy_documents) == 10
assert [doc.metadata['pdf_page'] for doc in policy_documents] == list(range(16, 26))
assert len({(doc.metadata['source_id'], doc.metadata['pdf_page'])
            for doc in policy_documents}) == 10
assert all(doc.page_content.strip() for doc in policy_documents)
assert all(doc.metadata['team_verified'] is False for doc in policy_documents)
assert all(doc.metadata['accepted_for_ingestion'] is False for doc in policy_documents)
assert all(doc.metadata['current_entitlement_answers_allowed'] is False
           for doc in policy_documents)
assert all(doc.metadata['current_benefit_availability'] == 'not_verified'
           for doc in policy_documents)
assert all('conditions_pdf_page' not in doc.metadata
           for doc in policy_documents if doc.metadata['pdf_page'] != 18)
assert policy_documents[2].page_content == policy_document.page_content
assert policy_documents[2].metadata == policy_document.metadata
print('Ten candidate page records checked. OCR pages and splitting remain pending.')


### Load one OCR proposal

This worked example loads the August corrigendum's physical page 2. Its text comes from the explicit `proposed_text` record; its citation still points to the original PDF. The proposal is AI-assisted text, not a completed human review.

Exercise 2 cell 9's `Document` and dictionary patterns are reused. Hash checks and the amendment fields are additions for this corpus. Read the OCR source's metadata afresh: copying the base policy's title, date or page-19 link would mislabel this document.

Page 1 contains the old wording, page 2 its replacement and page 3 the distribution list. This example retains the replacement role and the amended source/section; it does not implement answer selection. Run the earlier Stage 2 cells first. No model or API call is needed.


In [ ]:
ocr_review = json.loads(Path('data/ocr/maharashtra/review.json').read_text())
ocr_source = next(source for source in source_manifest['sources']
                  if source['source_id'] == 'maharashtra_corrigendum_2025-08-29')
ocr_page = next(page for page in ocr_review['pages']
                if page['source_id'] == ocr_source['source_id'] and page['pdf_page'] == 2)

assert ocr_page['source_pdf'] == ocr_source['filename']
assert ocr_page['source_sha256'] == ocr_source['sha256']
assert hashlib.sha256(Path(ocr_source['filename']).read_bytes()).hexdigest() == ocr_source['sha256']
assert 1 <= ocr_page['pdf_page'] <= ocr_source['pdf_page_count']

ocr_path = Path(ocr_page['proposed_text'])
ocr_bytes = ocr_path.read_bytes()
assert hashlib.sha256(ocr_bytes).hexdigest() == ocr_page['proposed_sha256']
ocr_text = ocr_bytes.decode('utf-8')
if not ocr_text.strip():
    raise ValueError(f'Empty OCR proposal: {ocr_path}')

ocr_metadata = {key: ocr_source[key] for key in (
    'state', 'policy_year', 'source_id', 'document_title', 'official_url',
    'document_date', 'accepted_for_ingestion', 'current_benefit_availability',
    'current_entitlement_answers_allowed', 'amends_source_id', 'amended_section'
)}
ocr_metadata.update({
    'source': ocr_source['filename'],
    'source_sha256': ocr_source['sha256'],
    'pdf_page': ocr_page['pdf_page'],
    'page_role': ocr_source['page_roles'][str(ocr_page['pdf_page'])],
    'team_verified': ocr_source['team_verified'] and ocr_page['team_verified'],
    'verification_cutoff': source_manifest['verification_cutoff'],
    'text_file': ocr_page['proposed_text'],
    'text_sha256': ocr_page['proposed_sha256'],
    'text_status': 'ai_proposal',
})
ocr_document = Document(page_content=ocr_text, metadata=ocr_metadata)

print(ocr_document.metadata)
print('OCR characters:', len(ocr_document.page_content))


In [ ]:
assert ocr_document.metadata['pdf_page'] == 2
assert ocr_document.metadata['page_role'] == ocr_page['page_role'] == 'replacement_wording_and_authority'
assert ocr_document.metadata['amends_source_id'] == policy_source['source_id']
assert ocr_document.metadata['amended_section'] == '4.2(1)'
assert ocr_document.metadata['source'].endswith('.pdf')
assert ocr_document.metadata['source'] != ocr_document.metadata['text_file']
assert 'conditions_pdf_page' not in ocr_document.metadata
assert ocr_document.metadata['team_verified'] is False
assert ocr_document.metadata['accepted_for_ingestion'] is False
assert ocr_document.metadata['current_entitlement_answers_allowed'] is False
assert ocr_document.metadata['current_benefit_availability'] == 'not_verified'
assert 'याऐवजी' in ocr_document.page_content
assert 'संबंधित विभाग /' in ocr_document.page_content
print('One OCR proposal checked against its recorded hashes and original PDF page.')


### Extend the examples

Team implementation continues here:

1. The English-page example produces ten records; the OCR example is a separate single document. Neither should be appended twice when assembling the complete list.
2. Generalize OCR loading to all eight explicit `proposed_text` records. Look up each record's own source, page and role. June/July lack the August-specific `amends_source_id`, `amended_section` and `page_roles` keys, so copying this example unchanged into a loop would fail. Carry June's `supplements_source_id` and July's `clarifies_source_id` where present.
3. Check 18 unique records before splitting. Preserve the August old/replacement/distribution roles and the cross-page relationships recorded in `data/ocr/maharashtra/TEXT_REVIEW.md`. Do not glob all `.txt` files: raw, proposed and future checked versions would be duplicated.
4. Adapt Lab 4 cell 77's splitter. Start with 1000 characters and 200 overlap; inspect table headings, rows, units, conditions and page-spanning clauses before accepting the settings.

Keep distribution-only text out of answer chunks and old wording distinguishable from its replacement. Manual source review and the team's two source-backed examples are deferred to the final review batch. See `STAGE_2_HANDOFF.md` for the complete technical checks. No index or answers are built in this stage.
